# Swing+ Results

In [ ]:
library(arrow)
library(tidyverse)
library(ggthemes)
library(IRdisplay)

ROOT  <- if (dir.exists(file.path(getwd(), "data"))) getwd() else dirname(getwd())
DATA  <- file.path(ROOT, "data")
PLOTS <- file.path(ROOT, "results", "plots")
dir.create(PLOTS, recursive = TRUE, showWarnings = FALSE)

FT_THEME <- theme_fivethirtyeight() +
  theme(axis.title        = element_text(),
        axis.text         = element_text(size = 10),
        panel.background  = element_rect(fill = "white", color = NA),
        plot.background   = element_rect(fill = "white", color = NA),
        legend.background = element_rect(fill = "white", color = NA),
        strip.text        = element_text(size = 12, face = "bold"),
        plot.title        = element_text(size = 16, face = "bold"))

plot_path <- function(name) {
  sub <- if (grepl("^swing_cards",       name)) "swing_cards"
    else if (grepl("^usage_heatmap",     name)) "usage_heatmap"
    else if (grepl("^repertoire",        name)) "repertoire"
    else if (grepl("^(swingplus|shape_breakdown)", name)) "swing_plus"
    else "predictiveness"
  d <- file.path(PLOTS, sub)
  dir.create(d, recursive = TRUE, showWarnings = FALSE)
  file.path(d, name)
}

## Aesthetic leaderboards (R · gt + headshots)

Swing+, Repertoire+, and Swing+-by-shape leaderboards rendered with **`gt` + `mlbplotR`** (MLBAM headshots, 538 theme) via `src/uncommited/leaderboard_table.R`, reading the same `data/*.parquet`. **Top 25 and Bottom 25** for each. The value column is colored **red (high) → blue (low)** with the domain fixed to the *full* qualified pool, so the Top 25 reads all-red and the Bottom 25 all-blue. The by-shape table reports each shape's **% of the hitter's swings** (usage share). PNGs saved to `results/plots/*_gt.png`.

In [ ]:
REGEN <- FALSE
if (REGEN) {
  system2("/usr/local/bin/Rscript", args = "src/uncommited/leaderboard_table.R", wait = TRUE)
}
for (png in c("swingplus_leaderboard_gt.png","swingplus_bottom_gt.png",
              "repertoire_leaderboard_gt.png","repertoire_bottom_gt.png",
              "swingplus_by_cluster_gt.png","swingplus_by_cluster_bottom_gt.png")) {
  p <- plot_path(png)
  if (file.exists(p)) IRdisplay::display_png(filename = p)
  else cat(sprintf("Not found: %s\n", png))
}

### Per-hitter breakdown

`shape_breakdown("name")` renders a single hitter's swing shapes (case-insensitive substring; a switch hitter returns both stances) ranked by Swing+, via the same R script in **breakdown mode** — headshot, archetype·situation, usage %, and Swing+. Color is the **league scale** (all qualified shapes), so a below-average hitter's shapes read pale/blue. Writes `results/plots/shape_breakdown_<name>_gt.png`.

In [ ]:
shape_breakdown <- function(name) {
  slug <- gsub("(^_|_$)", "", gsub("[^a-z0-9]+", "_", tolower(name)))
  system2("/usr/local/bin/Rscript",
          args = c("src/uncommited/leaderboard_table.R", name), wait = TRUE)
  p <- plot_path(sprintf("shape_breakdown_%s_gt.png", slug))
  IRdisplay::display_png(filename = p)
}
shape_breakdown("Arraez")

### Repertoire+ vs Swing+

Width and quality are independent axes — a wide repertoire says nothing about whether the swings are good. Scatter to confirm there's no clear relationship.

In [ ]:
rep_df <- read_parquet(file.path(DATA, "repertoire_scores.parquet"),
                       col_select = c("batter_id","batter_stand","repertoire_plus"))
ca     <- read_parquet(file.path(DATA, "cluster_assignments.parquet"),
                       col_select = c("play_id","batter_id","batter_stand"))
xrv    <- read_parquet(file.path(DATA, "xrv_swings.parquet"),
                       col_select = c("play_id","game_year","xrv_grade"))

sp <- ca |>
  inner_join(xrv, by = "play_id") |>
  filter(game_year %in% c(2024, 2025)) |>
  group_by(batter_id, batter_stand) |>
  summarise(swing_plus = mean(xrv_grade), .groups = "drop")

d     <- rep_df |> inner_join(sp, by = c("batter_id","batter_stand")) |> drop_na()
r_val <- cor(d$repertoire_plus, d$swing_plus)

plt <- ggplot(d, aes(repertoire_plus, swing_plus)) +
  geom_point(alpha = 0.45, size = 1.8, color = "#30a2da") +
  geom_smooth(method = "lm", se = FALSE, color = "#fc4f30", linewidth = 1.3) +
  annotate("text",
           x = min(d$repertoire_plus) + diff(range(d$repertoire_plus)) * 0.04,
           y = max(d$swing_plus)      - diff(range(d$swing_plus))      * 0.06,
           label = sprintf("r = %+.2f  (n = %d)", r_val, nrow(d)),
           hjust = 0, size = 4, color = "#fc4f30", fontface = "bold") +
  labs(title = "Repertoire breadth vs. swing quality",
       x = "Repertoire+", y = "Swing+") +
  FT_THEME

ggsave(plot_path("repertoire_vs_swingplus.png"), plt, width = 7, height = 6, dpi = 130)
print(plt)
cat(sprintf("r = %.3f, n = %d\n", r_val, nrow(d)))

## Validation

Four checks on the bespoke xRV / Swing+ pipeline:

1. **Year over year** — per-season agreement of predicted (`xrv`) and tables-realized (`realized_rv`) run value against the `delta_run_exp` anchor (models train on 2024–25; **2026 is held out**), then Swing+ reliability across consecutive seasons (is it a stable batter skill?).
2. **Expected vs actual** — batter-level scatter of mean predicted xRV against mean actual `delta_run_exp`.
3. **Feature importance** — gain per feature for each of the three models (`p_bip`, `p_foul`, `v_bip`), loaded from `data/xrv_models/` (produced by `src/xRV_model.py`).
4. **Predictive validity** — faceted scatter of Swing+ against real hitting outcomes (BA, OPS, wRC+, wOBA, WAR) from `season_stats_hitting`, one panel per metric with its Pearson r. *OPS+ is not stored in `mlb_db`; wRC+ is the park/league-adjusted analog and stands in for it.*

In [ ]:
val <- read_parquet(file.path(DATA, "xrv_swings.parquet"),
                    col_select = c("batter_id","batter_stand","game_year",
                                   "xrv","realized_rv","delta_run_exp","xrv_grade"))
cat(sprintf("%s swings | seasons %s\n",
            format(nrow(val), big.mark=","),
            paste(sort(unique(val$game_year)), collapse=", ")))

m <- val |> drop_na(delta_run_exp)
per_season <- m |>
  group_by(game_year) |>
  summarise(
    swings                  = n(),
    `corr(xRV, actual)`     = cor(xrv, delta_run_exp),
    `corr(realized, actual)`= cor(realized_rv, delta_run_exp),
    `rmse(realized, actual)`= sqrt(mean((delta_run_exp - realized_rv)^2)),
    `mean xRV`              = mean(xrv),
    .groups = "drop"
  ) |>
  mutate(across(where(is.double), \(x) round(x, 4)))

print(per_season)
cat("(2026 = held-out test season)\n")

In [ ]:
MIN_SW <- 200
sp <- val |>
  group_by(batter_id, game_year) |>
  summarise(swings = n(), swing_plus = mean(xrv_grade), .groups = "drop") |>
  filter(swings >= MIN_SW)

pairs <- list(c(2024, 2025), c(2025, 2026))
rel <- map_dfr(pairs, function(yrs) {
  wide <- sp |>
    filter(game_year %in% yrs) |>
    pivot_wider(id_cols = batter_id, names_from = game_year, values_from = swing_plus) |>
    drop_na()
  tibble(`season pair` = paste(yrs, collapse = " → "),
         batters       = nrow(wide),
         `Swing+ corr` = round(cor(wide[[as.character(yrs[1])]], wide[[as.character(yrs[2])]]), 3))
})
print(rel)

In [ ]:
library(xgboost)
library(patchwork)

model_dir   <- file.path(ROOT, "data", "xrv_models")
features    <- c("same_hand","plate_x_pull","plate_z_norm","pitch_type",
                 "swing_path_tilt","swing_length","bat_speed",
                 "vert_attack_angle","horz_attack_angle_pull")
shape_feats <- c("swing_path_tilt","swing_length","bat_speed",
                 "vert_attack_angle","horz_attack_angle_pull")
model_labels <- c(p_bip  = "P(in play)",
                  p_foul = "P(foul | not BIP)",
                  v_bip  = "E[batted-ball RV]")

get_gain <- function(model_name) {
  b   <- xgb.load(file.path(model_dir, paste0(model_name, ".json")))
  imp <- xgb.importance(model = b)
  tibble(feature = features) |>
    left_join(imp |> select(Feature, Gain), by = c("feature" = "Feature")) |>
    mutate(Gain    = replace_na(Gain, 0),
           pct     = 100 * Gain / sum(Gain),
           model   = model_name,
           is_shape = feature %in% shape_feats)
}

imp_long <- map_dfr(names(model_labels), get_gain) |>
  mutate(model = factor(model, levels = names(model_labels),
                        labels = model_labels))

plots <- map(levels(imp_long$model), function(m) {
  d <- imp_long |> filter(model == m) |> arrange(pct)
  ggplot(d, aes(x = pct, y = reorder(feature, pct), fill = is_shape)) +
    geom_col(alpha = 0.85) +
    scale_fill_manual(values = c("TRUE" = "#fc4f30", "FALSE" = "#30a2da"),
                      guide = "none") +
    labs(title = m, x = "% of total gain", y = NULL) +
    FT_THEME +
    theme(plot.title = element_text(size = 11))
})

combined <- wrap_plots(plots, nrow = 1) +
  plot_annotation(title = "xRV feature importance  (red = swing-shape, blue = context)",
                  theme = theme(plot.title = element_text(size = 14, face = "bold")))

out <- plot_path("xrv_feature_importance.png")
ggsave(out, combined, width = 15, height = 5, dpi = 130)
print(combined)
cat(sprintf("Saved → %s\n", out))